# In class 7 — molecules with RDKit

Twelve minutes, open notes, nothing to hand in but one number.

Everything here is from today's lecture, one step at a time. Each part shows you the
example we just ran, then asks you to do the same thing to a different molecule.

**How this works**
1. Run the **Setup** cell. It installs RDKit — about 30 seconds, and nothing below
   works without it.
2. For each part: run the example, then write one line in the empty cell under it.
3. The expected answer is in a `# ...` comment, so you can tell straight away whether
   you got it.
4. Run the **Check in** cell at the end and type its number into the CHEM 438 app.

Stuck for more than a minute? Ask — that is what the room is for.

<!-- c438-copy-note -->
> ### ▶️ First — make your own copy
> This notebook opened **read-only** (it comes straight from GitHub). To type your answers and keep your work:
>
> **File ▸ Save a copy in Drive**  —  or click **Copy to Drive** in the toolbar.
>
> Then work in *your* copy. It saves to your Google Drive automatically.

In [ ]:
# Setup — run this first
!pip install rdkit -q
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, Lipinski
print("RDKit is ready.")

---
## 1. A string becomes a molecule

`Chem.MolFromSmiles` takes text and gives you back a molecule you can measure.

In [ ]:
ethanol = Chem.MolFromSmiles("CCO")
print(ethanol.GetNumAtoms())    # 3

**Task 1.** Ibuprofen is `CC(C)Cc1ccc(cc1)C(C)C(=O)O`. Make it, and print its atom count.

In [ ]:
# your turn — expected: 15

Ibuprofen is C13H18O2, so that is 33 atoms in the formula but **15** here. RDKit leaves
hydrogens implicit — they are known from the structure, so it does not store them as
separate atoms.

---
## 2. The trap

A SMILES that RDKit cannot read does **not** raise an error. It hands back `None`, and
your code breaks later somewhere that has nothing to do with SMILES.

In [ ]:
good = Chem.MolFromSmiles("CCO")
bad  = Chem.MolFromSmiles("CCQ")      # Q is not an element
print(good, bad)                      # a Mol object, then None

RDKit does complain on its way out, in red — `SMILES Parse Error: ...`. That is a **log
message, not an exception.** Your program carried straight on, and `bad` is `None`. Which
is exactly why you have to check it yourself.

**Task 2.** Someone typos ibuprofen's last atom: `CC(C)Cc1ccc(cc1)C(C)C(=O)X`.
Print `True` if RDKit failed to read it — without reading the answer off the screen
yourself.

In [ ]:
# your turn — expected: True
# hint: compare the result to None

---
## 3. Draw it

In a notebook, a molecule drawn by `Draw.MolToImage` just appears.

In [ ]:
caffeine = Chem.MolFromSmiles("CN1C=NC2=C1C(=O)N(C(=O)N2C)C")
Draw.MolToImage(caffeine, size=(300, 300))

**Task 3.** Draw paracetamol (acetaminophen), `CC(=O)Nc1ccc(O)cc1`.

In [ ]:
# your turn — a benzene ring with an -OH on one side and an amide on the other

---
## 4. Measure it

Six numbers worth knowing. **Four of these six are impossible from a formula** — rings,
donors, acceptors and logP all need to know which atom is bonded to which.

In [ ]:
aspirin = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")

print("MW       ", round(Descriptors.MolWt(aspirin), 2))    # 180.16
print("logP     ", round(Descriptors.MolLogP(aspirin), 2))  # 1.31
print("TPSA     ", round(Descriptors.TPSA(aspirin), 1))     # 63.6
print("HBD      ", Lipinski.NumHDonors(aspirin))            # 1
print("HBA      ", Lipinski.NumHAcceptors(aspirin))         # 3
print("rings    ", Descriptors.RingCount(aspirin))          # 1

**Task 4.** Print the same six numbers for paracetamol.

In [ ]:
# your turn — expected: MW 151.16, logP 1.35, TPSA 49.3, HBD 2, HBA 2, rings 1

**Task 5.** Nicotine is `CN1CCCC1c1cccnc1`. Print its **hydrogen-bond donor** count.

In [ ]:
# your turn — expected: 0

Nicotine has two nitrogen atoms and **zero** donors. Neither nitrogen carries a hydrogen —
one is a tertiary amine, the other is in the aromatic ring. A formula says `C10H14N2` and
cannot tell you that; you need the bonds.

---
## 5. A real filter

Lipinski's rule of five: four thresholds, a rough guide to whether a molecule could work
as an oral drug. Note the `if mol is None` — part 2's rule, applied without being asked.

In [ ]:
def passes_lipinski(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return (Descriptors.MolWt(mol) <= 500
            and Descriptors.MolLogP(mol) <= 5
            and Lipinski.NumHDonors(mol) <= 5
            and Lipinski.NumHAcceptors(mol) <= 10)

print(passes_lipinski("CC(=O)Oc1ccccc1C(=O)O"))   # aspirin  -> True

**Task 6.** Cholesterol is `CC(C)CCCC(C)C1CCC2(C)C1CCC1C2CC=C2CC(O)CCC12C`.
Run `passes_lipinski` on it, then print the **one** descriptor that breaks a rule.

In [ ]:
# your turn — expected: False, because logP is 7.39 and the limit is 5
# (its MW is 386.66, donors 1, acceptors 1 — those three are all fine)

**Task 7.** Run the filter over all six molecules below and print a line for each.

In [ ]:
molecules = {
    "ibuprofen":   "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "paracetamol": "CC(=O)Nc1ccc(O)cc1",
    "nicotine":    "CN1CCCC1c1cccnc1",
    "vitamin C":   "OCC(O)C1OC(=O)C(O)=C1O",
    "cholesterol": "CC(C)CCCC(C)C1CCC2(C)C1CCC1C2CC=C2CC(O)CCC12C",
    "metformin":   "CN(C)C(=N)NC(N)=N",
}

# your turn — expected: every one True except cholesterol

> **A filter, not a law.** Read `False` as "look more carefully", never "throw it out".
> Cyclosporine is a drug real patients take and it breaks the weight rule outright.

---
## 6. Does it contain this group?

Hand over a **pattern** instead of a molecule: `MolFromSmarts`, not `MolFromSmiles`.
This is the query behind every "find compounds containing this group" search in every
chemical database.

In [ ]:
acid = Chem.MolFromSmarts("C(=O)[OH]")        # a carboxylic acid

print("aspirin ", Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O").HasSubstructMatch(acid))  # True
print("ethanol ", Chem.MolFromSmiles("CCO").HasSubstructMatch(acid))                    # False

**Task 8.** Vitamin C — **ascorbic acid** — is `OCC(O)C1OC(=O)C(O)=C1O`.

**Predict first**, out loud to the person next to you: does it contain a carboxylic acid?
Then check.

In [ ]:
# your turn

**`False`.** Ascorbic acid is an acid, and it has no carboxylic acid group — the acidic
hydrogen is on an enol `-OH` next to the ring's `C=O`. The name told you it was an acid;
only the structure tells you which kind. This is the whole argument for today's unit.

---
## Check in

Run the cell below and type the number it prints into the CHEM 438 app.

In [ ]:
molecules = {
    "ibuprofen":   "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "paracetamol": "CC(=O)Nc1ccc(O)cc1",
    "nicotine":    "CN1CCCC1c1cccnc1",
    "vitamin C":   "OCC(O)C1OC(=O)C(O)=C1O",
    "cholesterol": "CC(C)CCCC(C)C1CCC2(C)C1CCC1C2CC=C2CC(O)CCC12C",
    "metformin":   "CN(C)C(=N)NC(N)=N",
}
print("Your check-in value:",
      sum(1 for s in molecules.values() if passes_lipinski(s)))

---
## Done early?

1. Which of the six has the largest TPSA? (Vitamin C, at 107.2 — it is the one your gut
   would never pick.)
2. Metformin's logP is **-1.03**, the same as caffeine's. One is a diabetes drug and the
   other is in your coffee. What does that tell you about how much one descriptor can say?
3. Add a molecule of your own. Anything on the label of something in your bag — look its
   SMILES up on Wikipedia or PubChem and run it through the filter.